In [1]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
conn = sqlite3.connect("hospital.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS beds (
    bed_no INTEGER PRIMARY KEY,
    ward TEXT,
    bed_type TEXT,
    status TEXT
)
""")

conn.commit()
print("Database connected successfully!")

Database connected successfully!


In [3]:
cur.execute("SELECT COUNT(*) FROM beds")

if cur.fetchone()[0] == 0:
    beds = [
        (101, "General", "Normal", "Available"),
        (102, "General", "Normal", "Occupied"),
        (103, "ICU", "ICU", "Available"),
        (104, "ICU", "ICU", "Reserved"),
        (105, "Emergency", "Emergency", "Available")
    ]

    cur.executemany(
        "INSERT INTO beds VALUES (?, ?, ?, ?)", beds
    )
    conn.commit()

print("Bed records ready!")

Bed records ready!


In [4]:
cur.execute("""
CREATE TABLE IF NOT EXISTS history (
    bed_no INTEGER,
    old_status TEXT,
    new_status TEXT,
    changed_at TEXT
)
""")
conn.commit()

def show_data():
    df = pd.read_sql_query("SELECT * FROM beds", conn)
    table.delete(*table.get_children())

    for _, row in df.iterrows():
        table.insert("", "end", values=list(row))

    total = len(df)
    available = np.sum(df["status"] == "Available")
    occupied = np.sum(df["status"] == "Occupied")
    reserved = np.sum(df["status"] == "Reserved")

    dashboard.config(
        text=f"Total: {total} | Available: {available} | "
             f"Occupied: {occupied} | Reserved: {reserved}"
    )

    if available <= 1:
        alert.config(text="Warning: Only few beds are available!")
    else:
        alert.config(text="Bed availability is normal.")

    ward_info = []
    for ward in df["ward"].unique():
        part = df[df["ward"] == ward]
        percent = (sum(part["status"] == "Occupied") / len(part)) * 100
        ward_info.append(f"{ward}: {sum(part['status'] == 'Available')} available, "
                         f"{percent:.0f}% occupied")

    ward_label.config(text="Ward Summary: " + " | ".join(ward_info))

In [5]:
def add_bed():
    try:
        cur.execute(
            "INSERT INTO beds (bed_no, ward, bed_type, status) VALUES (?, ?, ?, ?)",
            (int(no.get()), ward.get(), bed_type.get(), status.get())
        )
        conn.commit()
        show_data()
        messagebox.showinfo("Success", "Bed added!")
    except ValueError:
        messagebox.showerror("Error", "Enter a valid bed number.")
    except sqlite3.IntegrityError:
        messagebox.showerror("Error", "Bed number already exists.")


def search_bed():
    value = search.get().strip()
    df = pd.read_sql_query(
        "SELECT * FROM beds WHERE CAST(bed_no AS TEXT) LIKE ? "
        "OR ward LIKE ? OR status LIKE ?",
        conn, params=(f"%{value}%", f"%{value}%", f"%{value}%")
    )
    table.delete(*table.get_children())
    for _, row in df.iterrows():
        table.insert("", "end", values=list(row))


def update_bed():
    try:
        bed_no = int(no.get())
        cur.execute("SELECT status FROM beds WHERE bed_no=?", (bed_no,))
        result = cur.fetchone()

        if not result:
            messagebox.showerror("Error", "Bed not found.")
            return

        old_status = result[0]
        new_status = status.get()

        cur.execute("UPDATE beds SET status=? WHERE bed_no=?",
                    (new_status, bed_no))

        if old_status != new_status:
            cur.execute(
                "INSERT INTO history VALUES (?, ?, ?, ?)",
                (bed_no, old_status, new_status,
                 datetime.now().strftime("%Y-%m-%d %H:%M"))
            )

        conn.commit()
        show_data()
        messagebox.showinfo("Success", "Status updated!")
    except ValueError:
        messagebox.showerror("Error", "Enter a valid bed number.")


def delete_bed():
    try:
        cur.execute("DELETE FROM beds WHERE bed_no=?", (int(no.get()),))
        conn.commit()
        show_data()
        messagebox.showinfo("Success", "Bed deleted if it existed.")
    except ValueError:
        messagebox.showerror("Error", "Enter a valid bed number.")

In [6]:
def suggest_bed():
    cur.execute(
        "SELECT bed_no, ward, bed_type FROM beds "
        "WHERE status='Available' AND ward=? AND bed_type=?",
        (ward.get(), bed_type.get())
    )
    result = cur.fetchone()

    if result:
        messagebox.showinfo(
            "Suggested Bed",
            f"Bed: {result[0]}\nWard: {result[1]}\nType: {result[2]}"
        )
    else:
        cur.execute(
            "SELECT bed_no, ward, bed_type FROM beds WHERE status='Available'"
        )
        alternatives = cur.fetchall()

        if alternatives:
            msg = "\n".join(
                f"Bed {b[0]} - {b[1]} ({b[2]})" for b in alternatives
            )
            messagebox.showinfo("Alternative Beds", msg)
        else:
            messagebox.showwarning("No Beds", "No available beds found.")


def emergency_finder():
    cur.execute(
        "SELECT bed_no, ward, bed_type FROM beds "
        "WHERE status='Available' AND ward IN ('Emergency', 'ICU')"
    )
    results = cur.fetchall()

    if results:
        msg = "\n".join(f"Bed {b[0]} - {b[1]}" for b in results)
        messagebox.showinfo("Emergency Beds", msg)
    else:
        messagebox.showwarning("No Beds", "No Emergency/ICU beds available.")


def show_history():
    df = pd.read_sql_query("SELECT * FROM history ORDER BY rowid DESC", conn)
    messagebox.showinfo("Bed Status History", df.to_string(index=False))


def daily_report():
    df = pd.read_sql_query("SELECT * FROM beds", conn)
    filename = "daily_bed_report.txt"

    with open(filename, "w", encoding="utf-8") as f:
        f.write("HOSPITAL BED DAILY REPORT\n")
        f.write(df.to_string(index=False))
        f.write("\n\nTotal Beds: " + str(len(df)))
        f.write("\nAvailable Beds: " +
                str(np.sum(df["status"] == "Available")))

    messagebox.showinfo("Report", f"Report saved as {filename}")

In [7]:
root = tk.Tk()
root.title("Hospital Bed Availability System")
root.geometry("1000x700")

tk.Label(
    root, text="HOSPITAL BED AVAILABILITY SYSTEM",
    font=("Arial", 18, "bold")
).pack(pady=10)

dashboard = tk.Label(root, font=("Arial", 12, "bold"))
dashboard.pack(pady=5)

alert = tk.Label(root, fg="red")
alert.pack()

ward_label = tk.Label(root, wraplength=900)
ward_label.pack(pady=5)

form = tk.Frame(root)
form.pack(pady=10)

tk.Label(form, text="Bed Number").grid(row=0, column=0)
no = tk.Entry(form)
no.grid(row=1, column=0, padx=5)

tk.Label(form, text="Ward").grid(row=0, column=1)
ward = ttk.Combobox(
    form, values=["General", "ICU", "Emergency"], state="readonly"
)
ward.grid(row=1, column=1, padx=5)
ward.set("General")

tk.Label(form, text="Bed Type").grid(row=0, column=2)
bed_type = ttk.Combobox(
    form, values=["Normal", "ICU", "Emergency"], state="readonly"
)
bed_type.grid(row=1, column=2, padx=5)
bed_type.set("Normal")

tk.Label(form, text="Status").grid(row=0, column=3)
status = ttk.Combobox(
    form, values=["Available", "Occupied", "Reserved"], state="readonly"
)
status.grid(row=1, column=3, padx=5)
status.set("Available")

tk.Button(form, text="Add Bed", command=add_bed).grid(row=2, column=0, pady=8)
tk.Button(form, text="Update Status", command=update_bed).grid(row=2, column=1)
tk.Button(form, text="Delete Bed", command=delete_bed).grid(row=2, column=2)

search_frame = tk.Frame(root)
search_frame.pack(pady=5)

search = tk.Entry(search_frame, width=25)
search.pack(side="left", padx=5)

tk.Button(search_frame, text="Search", command=search_bed).pack(side="left")
tk.Button(search_frame, text="Show All", command=show_data).pack(side="left", padx=5)

columns = ("Bed No", "Ward", "Bed Type", "Status")
table = ttk.Treeview(root, columns=columns, show="headings", height=10)

for col in columns:
    table.heading(col, text=col)
    table.column(col, width=180, anchor="center")

table.pack(fill="x", padx=15, pady=10)

buttons = tk.Frame(root)
buttons.pack(pady=10)

tk.Button(
    buttons, text="Suggest Bed", command=suggest_bed
).grid(row=0, column=0, padx=5)

tk.Button(
    buttons, text="Emergency Bed Finder", command=emergency_finder
).grid(row=0, column=1, padx=5)

tk.Button(
    buttons, text="Status History", command=show_history
).grid(row=0, column=2, padx=5)

tk.Button(
    buttons, text="Daily Report", command=daily_report
).grid(row=0, column=3, padx=5)

In [ ]:
# Available, Occupied and Reserved status colours
table.tag_configure("Available", foreground="green")
table.tag_configure("Occupied", foreground="red")
table.tag_configure("Reserved", foreground="orange")

# Display bed records with colours
def show_data():
    df = pd.read_sql_query("SELECT * FROM beds", conn)
    table.delete(*table.get_children())

    for _, row in df.iterrows():
        values = list(row)
        table.insert(
            "", "end",
            values=values,
            tags=(row["status"],)
        )

    total = len(df)
    available = np.sum(df["status"] == "Available")
    occupied = np.sum(df["status"] == "Occupied")
    reserved = np.sum(df["status"] == "Reserved")

    dashboard.config(
        text=f"Total: {total} | Available: {available} | "
             f"Occupied: {occupied} | Reserved: {reserved}"
    )

    alert.config(
        text="Warning: Few beds available!" if available <= 1 else
             "Bed availability is normal."
    )

    ward_info = []
    for name in df["ward"].unique():
        part = df[df["ward"] == name]
        busy = np.mean(part["status"] == "Occupied") * 100
        count = np.sum(part["status"] == "Available")
        ward_info.append(
            f"{name}: {count} available, {busy:.0f}% occupied"
        )

    ward_label.config(text="Ward Summary: " + " | ".join(ward_info))


show_data()
root.mainloop()